# Exercises XP: RAG with LangChain (Student)

## 0) Installation des bibliothèques


In [1]:
# On installe toutes les bibliothèques nécessaires au projet RAG.
# - datasets : pour charger le jeu de données Hugging Face
# - transformers : pour utiliser le petit modèle de langage (LLM) local
# - sentence-transformers : pour créer les embeddings (vecteurs) du texte
# - faiss-cpu : la base de données vectorielle qui stocke les embeddings
# - langchain* : le framework qui assemble toutes les briques du RAG
# - langchain-classic : contient RetrievalQA dans les versions récentes de LangChain
!pip -q install -U datasets transformers sentence-transformers faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from typing import List

# Pour charger le jeu de données Hugging Face
from datasets import load_dataset
# Pour créer le petit modèle de génération de texte (le LLM local)
from transformers import pipeline

# Document : la structure standard de LangChain pour stocker un texte + ses métadonnées (ex: sa source)
from langchain_core.documents import Document
# PromptTemplate : permet de construire des prompts réutilisables (pas utilisé directement ici mais utile à connaître)
from langchain_core.prompts import PromptTemplate
# RecursiveCharacterTextSplitter : découpe un long texte en petits morceaux (chunks)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# HuggingFaceEmbeddings : transforme un texte en vecteur numérique (embedding)
from langchain_community.embeddings import HuggingFaceEmbeddings
# FAISS : la base de données vectorielle qui stocke les embeddings et permet la recherche par similarité
from langchain_community.vectorstores import FAISS
# DistanceStrategy : la façon de mesurer la "distance" (similarité) entre deux vecteurs
from langchain_community.vectorstores.utils import DistanceStrategy

# HuggingFacePipeline : permet d'utiliser un pipeline Hugging Face comme un LLM LangChain
from langchain_huggingface import HuggingFacePipeline
# RetrievalQA : la chaîne LangChain qui combine retriever + LLM pour répondre aux questions (RAG)
from langchain_classic.chains import RetrievalQA

/tmp/ipykernel_960/2862104987.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


## 1) Charger le jeu de données et le convertir en Documents


In [3]:
from datasets import load_dataset

# Nom du jeu de données sur Hugging Face Hub
dataset_name = "m-ric/huggingface_doc"
# On ne prend que les 200 premières lignes pour que le notebook reste rapide à exécuter
split = "train[:200]"
# Nom de la colonne qui contient le texte principal du document
text_column = "text"
# Nom de la colonne qui indique d'où vient le texte (utile pour les citations et le débogage)
source_column = "source"

# On charge le jeu de données
ds = load_dataset(dataset_name, split=split)

# On regarde à quoi ressemblent les données brutes avant de les transformer
print("Dataset columns:", ds.column_names)
print("Example row:", ds[0])

# On convertit chaque ligne du dataset en objet Document de LangChain.
# page_content = le texte principal, metadata = les informations annexes (ici la source)
documents: List[Document] = []
for i, row in enumerate(ds):
    documents.append(
        Document(
            page_content=row[text_column],
            metadata={"source": row[source_column]})
    )

print("\nDocuments:", len(documents))
print("Example metadata:", documents[0].metadata)
print("Example page content (first 350 chars):\n", documents[0].page_content[:350])

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

huggingface_doc.csv:   0%|          | 0.00/22.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2647 [00:00<?, ? examples/s]

Dataset columns: ['text', 'source']
Example row: {'text': ' Create an Endpoint\n\nAfter your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. \n\n## 1. Enter the Hugging Face Repository ID and your desired endpoint name:\n\n<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-documentation/main/assets/1_repository.png" alt="select repository" />\n\n## 2. Select your Cloud Provider and region. Initially, only AWS will be available as a Cloud Provider with the `us-east-1` and `eu-west-1` regions. We will add Azure soon, and if you need to test Endpoints with other Cloud Providers or regions, please let us know.\n\n<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-documentation/main/assets/1

## 2) Fragmenter les documents (chunking)

On va essayer **deux réglages différents** de chunking, comme demandé dans la consigne, pour observer comment le nombre et le contenu des chunks changent.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- Réglage n°1 : chunks plus petits ---
# chunk_size = taille maximale (en caractères) d'un chunk
# chunk_overlap = nombre de caractères communs entre deux chunks consécutifs (évite de couper une idée en deux)
chunk_size_v1 = 500  # Un bon point de départ pour chunk_size est 500
chunk_overlap_v1 = 50  # Un bon point de départ pour chunk_overlap est 50

splitter_v1 = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size_v1,
    chunk_overlap=chunk_overlap_v1
)

splits_v1 = splitter_v1.split_documents(documents)
print("[Réglage 1] Chunks:", len(splits_v1))
print("[Réglage 1] First chunk metadata:", splits_v1[0].metadata)
print("[Réglage 1] First chunk content (first 350 chars):\n", splits_v1[0].page_content[:350])

[Réglage 1] Chunks: 5966
[Réglage 1] First chunk metadata: {'source': 'huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx'}
[Réglage 1] First chunk content (first 350 chars):
 Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 




In [5]:
# --- Réglage n°2 : chunks plus grands, avec plus de chevauchement ---
# On double la taille des chunks et on augmente le chevauchement pour comparer.
chunk_size_v2 = 1000
chunk_overlap_v2 = 100

splitter_v2 = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size_v2,
    chunk_overlap=chunk_overlap_v2
)

splits_v2 = splitter_v2.split_documents(documents)
print("[Réglage 2] Chunks:", len(splits_v2))
print("[Réglage 2] First chunk metadata:", splits_v2[0].metadata)
print("[Réglage 2] First chunk content (first 350 chars):\n", splits_v2[0].page_content[:350])

# Observation attendue : avec des chunks plus grands (Réglage 2), on obtient MOINS de chunks au total,
# mais chaque chunk contient PLUS de contexte. Avec des chunks plus petits (Réglage 1), on obtient PLUS
# de chunks, plus précis mais parfois moins riches en contexte.

# Pour la suite du notebook, on garde le Réglage 1 (500/50), qui est un bon compromis pour ce jeu de données.
splits = splits_v1

[Réglage 2] Chunks: 2731
[Réglage 2] First chunk metadata: {'source': 'huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx'}
[Réglage 2] First chunk content (first 350 chars):
 Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 




## 3) Vector store + retriever (FAISS)


In [6]:
# On choisit un modèle d'embedding léger et rapide (transforme un texte en vecteur numérique)
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

# On construit la base vectorielle FAISS à partir de nos chunks et de leurs embeddings.
# On utilise la stratégie de distance COSINE, adaptée à ce type d'embeddings.
vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings,
    distance_strategy=DistanceStrategy.COSINE
)

# Le retriever est l'objet qui, à partir d'une question, va chercher les k chunks les plus proches (les plus pertinents)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever ready")

/tmp/ipykernel_960/2352238425.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=embedding_model)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retriever ready


### Comparer différentes valeurs de k

On teste plusieurs valeurs de `k` (le nombre de chunks récupérés) sur une même question, pour voir comment cela change la pertinence et la couverture des résultats.

In [7]:
test_query = "How can I retrieve a model from the Hugging Face Hub?"

# On compare k=2 (peu de résultats, très ciblés), k=4 (valeur par défaut) et k=6 (plus de résultats, plus de couverture)
for k_value in [2, 4, 6]:
    temp_retriever = vectorstore.as_retriever(search_kwargs={"k": k_value})
    results = temp_retriever.invoke(test_query)
    print(f"\n=== k = {k_value} ({len(results)} chunks récupérés) ===")
    for doc in results:
        print("-", doc.metadata.get("source"))

# Observation attendue : plus k est grand, plus on récupère de chunks (donc plus de "couverture"),
# mais certains chunks supplémentaires peuvent être moins pertinents (donc la pertinence moyenne peut baisser).


=== k = 2 (2 chunks récupérés) ===
- huggingface/blog/blob/main/fastai.md
- huggingface/transformers/blob/main/docs/source/en/model_sharing.md

=== k = 4 (4 chunks récupérés) ===
- huggingface/blog/blob/main/fastai.md
- huggingface/transformers/blob/main/docs/source/en/model_sharing.md
- huggingface/blog/blob/main/fastai.md
- huggingface/transformers/blob/main/docs/source/en/model_sharing.md

=== k = 6 (6 chunks récupérés) ===
- huggingface/blog/blob/main/fastai.md
- huggingface/transformers/blob/main/docs/source/en/model_sharing.md
- huggingface/blog/blob/main/fastai.md
- huggingface/transformers/blob/main/docs/source/en/model_sharing.md
- huggingface/blog/blob/main/fastai.md
- huggingface/blog/blob/main/fastai.md


## 4) Contrôle de santé mentale de la récupération (obligatoire)

Avant de générer des réponses avec le LLM, on vérifie **manuellement** que le retriever renvoie bien des chunks pertinents pour une question donnée. C'est une étape essentielle pour déboguer un système RAG : si les chunks récupérés ne sont pas pertinents, la réponse finale du LLM sera mauvaise, même si le LLM lui-même fonctionne bien.

In [8]:
# On choisit une question de test
sanity_query = "How can I retrieve a model from the Hugging Face Hub?"

# On récupère les chunks les plus proches de la question avec le retriever final (k=4)
retrieved_chunks = retriever.invoke(sanity_query)

print(f"Question testée : {sanity_query}\n")
print(f"Nombre de chunks récupérés : {len(retrieved_chunks)}\n")

# On affiche chaque chunk en entier pour pouvoir juger "à l'oeil" s'il est pertinent ou non
for i, chunk in enumerate(retrieved_chunks, start=1):
    print(f"--- Chunk {i} (source: {chunk.metadata.get('source')}) ---")
    print(chunk.page_content)
    print()

# Question à se poser en tant que débutant :
# - Est-ce que ces chunks contiennent réellement une réponse à la question posée ?
# - Si NON : il faut ajuster chunk_size / chunk_overlap (revenir à la Section 2) et/ou augmenter la valeur de k
#   (revenir à la Section 3), puis refaire ce contrôle avant de continuer.
# - Si OUI : on peut passer à la suite en toute confiance, la récupération fonctionne correctement.

Question testée : How can I retrieve a model from the Hugging Face Hub?

Nombre de chunks récupérés : 4

--- Chunk 1 (source: huggingface/blog/blob/main/fastai.md) ---
## Joining Hugging Face and installation

To share models in the Hub, you will need to have a user. Create it on the [Hugging Face website](https://huggingface.co/join).

--- Chunk 2 (source: huggingface/transformers/blob/main/docs/source/en/model_sharing.md) ---
Now when you navigate to your Hugging Face profile, you should see your newly created model repository. Clicking on the **Files** tab will display all the files you've uploaded to the repository.

For more details on how to create and upload files to a repository, refer to the Hub documentation [here](https://huggingface.co/docs/hub/how-to-upstream).

## Upload with the web interface

--- Chunk 3 (source: huggingface/blog/blob/main/fastai.md) ---
This integration is made possible by the [`huggingface_hub`](https://github.com/huggingface/huggingface_hub) library.

## 5) Construire la chaîne RAG


In [11]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA  # on latest stack

# Petit modèle de langage local, léger, qui ne nécessite pas de clé API
llm_id = "google/flan-t5-small"

# On crée le pipeline Hugging Face. Comme flan-t5 est un modèle "text-to-text" (il transforme un texte
# d'entrée en texte de sortie), on utilise la tâche "text-generation".
hf = pipeline(
    task="text-generation",
    model=llm_id,
    max_new_tokens=256
)

# On enveloppe le pipeline Hugging Face dans un objet LangChain, pour pouvoir l'utiliser dans une chaîne RAG
llm = HuggingFacePipeline(pipeline=hf)

# RetrievalQA assemble automatiquement : question -> retriever (récupère les chunks) -> llm (génère la réponse)
# chain_type="stuff" veut dire qu'on "empile" simplement tous les chunks récupérés dans le prompt du LLM.
# C'est la méthode la plus simple, bien adaptée à un petit nombre de chunks courts (k=4 ici).
# return_source_documents=True permet de récupérer les sources utilisées pour la réponse (utile pour vérifier le RAG).
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True
)

print("RAG chain ready")

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', '

RAG chain ready


## 6) Démo : RAG vs no-RAG


In [ ]:
questions = [
    "How can I retrieve a model from the Hugging Face Hub?",
    "What is a pipeline in the Transformers library?",
    "How do I push a model to the Hugging Face Hub?",
]

for q in questions:
    # --- Sans RAG (le LLM répond uniquement avec ses connaissances internes, sans consulter le jeu de données) ---
    no_rag_prompt = (
        "Answer the question. If you are not sure, say you are not sure.\n\n"
        f"Question: {q}\n"
        "Answer:"
    )
    no_rag_answer = hf(no_rag_prompt)[0]["generated_text"]

    # --- Avec RAG (le LLM répond en s'appuyant sur les chunks récupérés dans le jeu de données) ---
    # RetrievalQA attend un dictionnaire avec la clé "query" contenant la question posée.
    rag_result = qa({"query": q})

    print("=" * 80)
    print("Q:", q)
    print("\nNo-RAG answer:\n", no_rag_answer)
    print("\nRAG answer:\n", rag_result["result"])
    print("\nSources:")
    for d in rag_result["source_documents"]:
        print("-", d.metadata.get("source"))
    print()

# Ce qu'il faut observer, pour chaque question :
# - La réponse "No-RAG" vient uniquement de la mémoire du modèle : elle peut être vague, incomplète ou fausse.
# - La réponse "RAG" s'appuie sur de vrais extraits du jeu de données (affichés dans "Sources"),
#   ce qui la rend généralement plus fiable et vérifiable.
# - Si pour une question donnée les sources ne semblent pas pertinentes, revenir à la Section 4 (contrôle de
#   la récupération) pour ajuster chunk_size / chunk_overlap et/ou k avant de conclure sur la qualité du LLM.
